### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [ ]:
%pip install scikeras

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [24]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons = 32,layers = 1):
    model = Sequential()
    model.add(Dense(neurons,activation='relu',input_shape = (X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [25]:
### create keras classifier

model = KerasClassifier(build_fn=create_model,neurons=32, layers=1,verbose=1)

In [26]:
# Define the grid search parameters
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [27]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 744us/step - accuracy: 0.7369 - loss: 0.5557
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step - accuracy: 0.7973 - loss: 0.4593
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 462us/step - accuracy: 0.8069 - loss: 0.4377
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 480us/step - accuracy: 0.8134 - loss: 0.4254
Epoch 5/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 569us/step - accuracy: 0.8228 - loss: 0.4152
Epoch 6/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 580us/step - accuracy: 0.8281 - loss: 0.4055
Epoch 7/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 545us/step - accuracy: 0.8305 - loss: 0.3966
Epoch 8/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 538us/step - accuracy: 0.8389 - loss: 0.3880
Epoch 9/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 483us/step - accuracy: 0.8423 - loss: 0.3798
Epoch 10/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 471us/step - accuracy: 0.8446 - loss: 0.3731
Epoch 11/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 466us/step - accuracy: 0.8470 - loss: 0.3673
Epoch 12/50
167/167 ━━━━━━━━━━

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 538us/step - accuracy: 0.5593 - loss: 0.7141
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 459us/step - accuracy: 0.8020 - loss: 0.4973
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 455us/step - accuracy: 0.8116 - loss: 0.4532
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 461us/step - accuracy: 0.8164 - loss: 0.4358
Epoch 5/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 474us/step - accuracy: 0.8170 - loss: 0.4245
Epoch 6/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 446us/step - accuracy: 0.8230 - loss: 0.4139
Epoch 7/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 454us/step - accuracy: 0.8303 - loss: 0.4040
Epoch 8/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 447us/step - accuracy: 0.8386 - loss: 0.3949
Epoch 9/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 459us/step - accuracy: 0.8442 - loss: 0.3863
Epoch 10/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 446us/step - accuracy: 0.8453 - loss: 0.3789
Epoch 11/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 455us/step - accuracy: 0.8476 - loss: 0.3729
Epoch 12/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


... (89 outputs left out)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 862us/step - accuracy: 0.8133 - loss: 0.4381
Epoch 2/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 579us/step - accuracy: 0.8519 - loss: 0.3628
Epoch 3/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 585us/step - accuracy: 0.8607 - loss: 0.3412
Epoch 4/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 592us/step - accuracy: 0.8648 - loss: 0.3331
Epoch 5/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 590us/step - accuracy: 0.8646 - loss: 0.3268
Epoch 6/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 589us/step - accuracy: 0.8639 - loss: 0.3231
Epoch 7/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 601us/step - accuracy: 0.8693 - loss: 0.3170
Epoch 8/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 573us/step - accuracy: 0.8697 - loss: 0.3121
Epoch 9/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 567us/step - accuracy: 0.8744 - loss: 0.3089
Epoch 10/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 582us/step - accuracy: 0.8725 - loss: 0.3082
Epoch 11/100
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 601us/step - accuracy: 0.8744 - loss: 0.3027
Epoch 12/100
167/167 ━━━━━━━━━━

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 463us/step - accuracy: 0.7789 - loss: 0.5006
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 427us/step - accuracy: 0.8127 - loss: 0.4324
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 443us/step - accuracy: 0.8288 - loss: 0.4058
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 416us/step - accuracy: 0.8444 - loss: 0.3803
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 428us/step - accuracy: 0.8524 - loss: 0.3621
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 435us/step - accuracy: 0.8556 - loss: 0.3527
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 424us/step - accuracy: 0.8571 - loss: 0.3466
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 419us/step - accuracy: 0.8616 - loss: 0.3434
Epoch 9/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 465us/step - accuracy: 0.8610 - loss: 0.3400
Epoch 10/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 414us/step - accuracy: 0.8616 - loss: 0.3377
Epoch 11/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 471us/step - accuracy: 0.8620 - loss: 0.3362
Epoch 12/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 